# Objetivo_2

In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
import altair as alt
import datos
import plotly.graph_objects as go
import warnings
import plotly.subplots as sp

warnings.simplefilter(action='ignore', category=FutureWarning)

tipo de afiliación, lugar de nacimiento/procedencia, escolaridad,   antecedente personal de dislipidemia o hipercolesterolemia, de hipertensión, de infarto de miocardio de diabetes, insuficiencia cardiaca congestiva, nefropatía, arritmia, válvula, angeopatía, fumador y bebedor.  

+ IMC_calculada
+ perimetro_abdominal
+ presion_arterial_sistolica
+ presion_arterial_diastolica

+ <19.9 bajo peso, 20-24.9 normo peso, 25-29.9 sobrepeso, >=30 obesidad

+ normal (diastólica <80 y sistólica <120), 
 elevada (sistólica 120-129 y diastólica <80),
 alta nivel 1 (sistólica 130-139 o diastólica 80-89), 
 alta nivel 2 (sistólica >=140, diastólica >=90), 
 crisis hipertensiva (>180 diastólica y/o >120 sistólica)
+ aumentado >94 hombres, >80 mujeres; incremento sustancial >102 hombres y >88 en mujeres

### IMC

In [9]:

intervalos = {
    'IMC': [0, 19.9, 24.9, 29.9, 40, max(datos.datos['IMC'])+1]
}

# Agrupaciones disponibles
agrupaciones = [
    'sexo',
    'tipo_de_afiliacion',
    'escolaridad',
    'diabetes mellitus',
    'hipercolesterolemia',
    'dislipidemia',
    'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2'
]
def plot_all():
    fig = px.box(datos.datos, y='IMC', title='Boxplot crudo del IMC')
    fig.update_layout(xaxis_title='IMC')
    fig.show()
    fig = px.histogram(datos.datos, x='IMC', nbins=10, title='Histograma crudo del IMC', marginal='rug')
    fig.update_layout(xaxis_title='IMC')
    fig.show()

    # Mostrar tabla descriptiva general
    tabla = datos.datos['IMC'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas del IMC')
    fig.show()

    # Graficar IMC por cada agrupación, excluyendo 'lugar_de_procedencia' y 'lugar_de_nacimiento'
    for agrupacion in agrupaciones:
        if agrupacion in ['tipo_de_afiliacion','escolaridad']:
            continue

        # Estadísticas descriptivas por agrupación
        tabla2 = datos.datos.groupby(agrupacion)['IMC'].describe().reset_index()
        tabla2 = tabla2.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
        fig = go.Figure(data=[go.Table(
            header=dict(values=list(tabla2.columns), fill_color='paleturquoise', align='left'),
            cells=dict(values=[tabla2[col] for col in tabla2.columns], fill_color='lavender', align='left')
        )])
        fig.update_layout(title_text=f'Estadísticas Descriptivas del IMC por {agrupacion}')
        fig.show()

        # Boxplot y histograma por agrupación
        fig = px.box(datos.datos, x='IMC', y=agrupacion, title=f'Boxplot del IMC por {agrupacion}')
        fig.update_layout(xaxis_title='IMC')
        fig.show()

        fig = px.histogram(datos.datos, x='IMC', color=agrupacion, nbins=10, title=f'Histograma del IMC por {agrupacion}', marginal='rug')
        fig.update_layout(xaxis_title='IMC')
        fig.show()


    
    datos.datos['IMC_intervalo'] = pd.cut(datos.datos['IMC'], bins=[0, 19.9, 24.9, 29.9, 40.0, 50.66])
    datos.datos['IMC_intervalo'] = datos.datos['IMC_intervalo'].astype(str)

        
    tabla3 = datos.datos.groupby('IMC_intervalo')['IMC'].describe().reset_index()
    tabla3 = tabla3.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla3.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla3[col] for col in tabla3.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas del IMC por Intervalos')
    fig.show()

    orden_intervalos = ['(0.0, 19.9]', '(19.9, 24.9]', '(24.9, 29.9]', '(29.9, 40.0]', '(40.0, 50.66]']
    fig = px.box(datos.datos, x=pd.Categorical(datos.datos['IMC_intervalo'], categories=orden_intervalos, ordered=True), 
                y='IMC', title="Boxplot del IMC por Intervalos", color_discrete_sequence=["blue"])
    fig.update_layout(xaxis_title='Intervalo IMC', yaxis_title='IMC', xaxis={'categoryorder':'array', 'categoryarray':orden_intervalos})
    fig.show()
plot_all()
for agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento']:
    tabla_freq = datos.datos[agrupacion].value_counts().reset_index()
    tabla_freq.columns = [agrupacion, 'Frecuencia']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_freq.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_freq[col] for col in tabla_freq.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text=f'Tabla de Frecuencias para {agrupacion}')
    fig.show()


### Perimetro abdominal

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Definir los intervalos para perimetro_abdominal
intervalos = {
    'masculino': [0, 94, 102, max(datos.datos['perimetro_abdominal'])],
    'femenino': [0, 80, 88, max(datos.datos['perimetro_abdominal'])] 
}
agrupaciones = [
        'sexo', 'tipo_de_afiliacion', 'lugar_de_procedencia',
        'lugar_de_nacimiento', 'escolaridad', 'diabetes mellitus',
        'hipercolesterolemia', 'dislipidemia'
    ]

def plot_all():
    # Graficar perimetro_abdominal crudo
    fig = px.box(datos.datos, y='perimetro_abdominal', title='Boxplot crudo de perimetro_abdominal')
    fig.update_layout(yaxis_title='Perímetro Abdominal')
    fig.show()

    fig = px.histogram(datos.datos, x='perimetro_abdominal', nbins=10, title='Histograma crudo de perimetro_abdominal', marginal='rug')
    fig.update_layout(xaxis_title='Perímetro Abdominal')
    fig.show()

    # Mostrar tabla descriptiva general verticalmente
    tabla = datos.datos['perimetro_abdominal'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas del perimetro_abdominal')
    fig.show()

    # Agrupaciones
    agrupaciones = [
        'sexo', 'tipo_de_afiliacion',  'escolaridad', 'diabetes mellitus',
        'hipercolesterolemia', 'dislipidemia', 'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2'
    ]
    # 'lugar_de_procedencia',
    #    'lugar_de_nacimiento',

    for agrupacion in agrupaciones:
        if agrupacion not in ['lugar_de_procedencia', 'lugar_de_nacimiento','tipo_de_afiliacion','escolaridad']:
            # Graficar perimetro_abdominal por la variable de agrupación
            fig = px.box(datos.datos, x='perimetro_abdominal', y=agrupacion, title=f'Boxplot de perimetro_abdominal por {agrupacion}')
            fig.update_layout(xaxis_title='Perímetro Abdominal')
            fig.show()

            fig = px.histogram(datos.datos, x='perimetro_abdominal', color=agrupacion, nbins=10, title=f'Histograma de perimetro_abdominal por {agrupacion}', marginal='rug')
            fig.update_layout(xaxis_title='Perímetro Abdominal')
            fig.show()

            # Mostrar tabla descriptiva por grupo verticalmente
            tabla2 = datos.datos.groupby(agrupacion)['perimetro_abdominal'].describe().reset_index()
            tabla2 = tabla2.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
            fig = go.Figure(data=[go.Table(
                header=dict(values=list(tabla2.columns), fill_color='paleturquoise', align='left'),
                cells=dict(values=[tabla2[col] for col in tabla2.columns], fill_color='lavender', align='left')
            )])
            fig.update_layout(title_text=f'Estadísticas Descriptivas del perimetro_abdominal por {agrupacion}')
            fig.show()
    hom = datos.datos[datos.datos['sexo'] == 'masculino']
    muj = datos.datos[datos.datos['sexo'] == 'femenino']
    hom['Intervalo'] = pd.cut(hom['perimetro_abdominal'], bins=[0, 94, 102, hom['perimetro_abdominal'].max()], labels=['Normal', 'Aumentado', 'Incremento Sustancial'])
    muj['Intervalo'] = pd.cut(muj['perimetro_abdominal'], bins=[0, 80, 88, muj['perimetro_abdominal'].max()], labels=['Normal', 'Aumentado', 'Incremento Sustancial'])
    fig_hombres = go.Figure()
    fig_hombres.add_trace(go.Box(y=hom[hom['Intervalo'] == 'Normal']['perimetro_abdominal'], name='Normal', marker_color='green'))
    fig_hombres.add_trace(go.Box(y=hom[hom['Intervalo'] == 'Aumentado']['perimetro_abdominal'], name='Aumentado', marker_color='orange'))
    fig_hombres.add_trace(go.Box(y=hom[hom['Intervalo'] == 'Incremento Sustancial']['perimetro_abdominal'], name='Incremento Sustancial', marker_color='red'))
    fig_hombres.update_layout(title="Gráficos de Caja del Perímetro Abdominal - Hombres", yaxis_title="Perímetro Abdominal (cm)", boxmode='group', xaxis_title="Intervalos")
    fig_hombres.show()
    fig_mujeres = go.Figure()
    fig_mujeres.add_trace(go.Box(y=muj[muj['Intervalo'] == 'Normal']['perimetro_abdominal'], name='Normal', marker_color='blue'))
    fig_mujeres.add_trace(go.Box(y=muj[muj['Intervalo'] == 'Aumentado']['perimetro_abdominal'], name='Aumentado', marker_color='purple'))
    fig_mujeres.add_trace(go.Box(y=muj[muj['Intervalo'] == 'Incremento Sustancial']['perimetro_abdominal'], name='Incremento Sustancial', marker_color='darkred'))
    fig_mujeres.update_layout(title="Gráficos de Caja del Perímetro Abdominal - Mujeres", yaxis_title="Perímetro Abdominal (cm)", boxmode='group', xaxis_title="Intervalos")
    fig_mujeres.show()
plot_all()

C:\Users\loren\AppData\Local\Temp\ipykernel_15056\2705330239.py:78: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\loren\AppData\Local\Temp\ipykernel_15056\2705330239.py:79: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### POC_hba1

In [11]:

intervalos = {
    'masculino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])],
    'femenino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])]
}

def plot_all():
    fig = px.box(datos.datos, y='presion_arterial_sistolica', title='Boxplot crudo de presión arterial sistólica')
    fig.update_layout(yaxis_title='Presión Arterial Sistólica')
    fig.show()

    fig = px.histogram(datos.datos, x='presion_arterial_sistolica', nbins=10, title='Histograma crudo de presión arterial sistólica', marginal='rug')
    fig.update_layout(xaxis_title='Presión Arterial Sistólica')
    fig.show()

    tabla = datos.datos['presion_arterial_sistolica'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas de presión arterial sistólica')
    fig.show()

    agrupaciones = ['sexo', 'tipo_de_afiliacion', 'lugar_de_procedencia', 'lugar_de_nacimiento', 'escolaridad', 'diabetes mellitus', 'hipercolesterolemia', 'dislipidemia']
    for agrupacion in agrupaciones:
        if agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento', 'tipo_de_afiliacion', 'escolaridad']:
            continue
        else:
            fig = px.box(datos.datos, x='presion_arterial_sistolica', y=agrupacion, title=f'Boxplot de presión arterial sistólica por {agrupacion}')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica', yaxis_title=agrupacion)
            fig.show()

            fig = px.histogram(datos.datos, x='presion_arterial_sistolica', color=agrupacion, nbins=10, title=f'Histograma de presión arterial sistólica por {agrupacion}', marginal='rug')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica')
            fig.show()

            tabla_agrupacion = datos.datos.groupby(agrupacion)['presion_arterial_sistolica'].describe().reset_index()
            tabla_agrupacion = tabla_agrupacion.rename(columns={'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 'min': 'Mínimo', 'max': 'Máximo'})
            fig = go.Figure(data=[go.Table(
                header=dict(values=list(tabla_agrupacion.columns), fill_color='paleturquoise', align='left'),
                cells=dict(values=[tabla_agrupacion[col] for col in tabla_agrupacion.columns], fill_color='lavender', align='left')
            )])
            fig.update_layout(title_text=f'Estadísticas Descriptivas de presión arterial sistólica por {agrupacion}')
            fig.show()

    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(datos.datos['presion_arterial_sistolica'], intervalos['masculino'], right=False).astype(str)
    tabla_intervalos = datos.datos.groupby('presion_arterial_sistolica_intervalo')['presion_arterial_sistolica'].describe().reset_index()
    tabla_intervalos.columns = ['Intervalo', 'Cuenta', 'Media', 'Desviación estándar', 'Mínimo', '25%', 'Mediana', '75%', 'Máximo']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_intervalos.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_intervalos[col] for col in tabla_intervalos.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas por Intervalos de presión arterial sistólica')
    fig.show()

    intervalos_masculino = [0, 120, 130, 139, float('inf')]
    labels = ['[0, 120)', '[120, 130)', '[130, 139)', '[139, inf)']
    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(datos.datos['presion_arterial_sistolica'], bins=intervalos_masculino, labels=labels, right=False, include_lowest=True)
    datos_filtrados = datos.datos.dropna(subset=['presion_arterial_sistolica_intervalo'])
    datos_filtrados['presion_arterial_sistolica_intervalo'] = pd.Categorical(datos_filtrados['presion_arterial_sistolica_intervalo'], categories=labels, ordered=True)
    fig = px.box(datos_filtrados, x='presion_arterial_sistolica_intervalo', y='presion_arterial_sistolica', title='Boxplot de Presión Arterial Sistólica por Intervalos', color_discrete_sequence=["blue"])
    fig.update_layout(xaxis_title='Intervalo', yaxis_title='Presión Arterial Sistólica', xaxis={'categoryorder': 'array', 'categoryarray': labels})
    fig.show()

plot_all()

C:\Users\loren\AppData\Local\Temp\ipykernel_15056\1668242866.py:60: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



### Presion Arterial

####   Presion arterial sistolica

In [12]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Definición de intervalos para presión arterial sistólica
intervalos = {
    'masculino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])],
    'femenino': [0, 120, 130, 139, max(datos.datos['presion_arterial_sistolica'])]
}

def plot_all():
    # Boxplot crudo
    fig = px.box(datos.datos, y='presion_arterial_sistolica', title='Boxplot crudo de presión arterial sistólica')
    fig.update_layout(yaxis_title='Presión Arterial Sistólica')
    fig.show()

    # Histograma crudo
    fig = px.histogram(datos.datos, x='presion_arterial_sistolica', nbins=10, 
                       title='Histograma crudo de presión arterial sistólica', marginal='rug')
    fig.update_layout(xaxis_title='Presión Arterial Sistólica')
    fig.show()

    # Tabla descriptiva general
    tabla = datos.datos['presion_arterial_sistolica'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas de presión arterial sistólica')
    fig.show()

    # Tablas descriptivas y gráficos por agrupaciones
    agrupaciones = [
        'sexo', 'tipo_de_afiliacion',
        'escolaridad', 'diabetes mellitus', 'hipercolesterolemia', 'dislipidemia',
        'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2'
    ]
    for agrupacion in agrupaciones:
        if agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento', 'tipo_de_afiliacion', 'escolaridad']:
            continue  # Omitir estas agrupaciones específicas
        else:
            # Boxplot por agrupación
            fig = px.box(datos.datos, x='presion_arterial_sistolica', y=agrupacion, 
                         title=f'Boxplot de presión arterial sistólica por {agrupacion}')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica', yaxis_title=agrupacion)
            fig.show()

            # Histograma por agrupación
            fig = px.histogram(datos.datos, x='presion_arterial_sistolica', color=agrupacion, nbins=10, 
                               title=f'Histograma de presión arterial sistólica por {agrupacion}', marginal='rug')
            fig.update_layout(xaxis_title='Presión Arterial Sistólica')
            fig.show()

            # Tabla descriptiva por agrupación
            tabla_agrupacion = datos.datos.groupby(agrupacion)['presion_arterial_sistolica'].describe().reset_index()
            tabla_agrupacion = tabla_agrupacion.rename(columns={
                'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 
                'min': 'Mínimo', 'max': 'Máximo'
            })
            fig = go.Figure(data=[go.Table(
                header=dict(values=list(tabla_agrupacion.columns), fill_color='paleturquoise', align='left'),
                cells=dict(values=[tabla_agrupacion[col] for col in tabla_agrupacion.columns], fill_color='lavender', align='left')
            )])
            fig.update_layout(title_text=f'Estadísticas Descriptivas de presión arterial sistólica por {agrupacion}')
            fig.show()

    # Tabla descriptiva por intervalos
    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_sistolica'],
        intervalos['masculino'],  # Usar intervalos para masculino como ejemplo
        right=False
    ).astype(str)
    tabla_intervalos = datos.datos.groupby('presion_arterial_sistolica_intervalo')['presion_arterial_sistolica'].describe().reset_index()
    tabla_intervalos.columns = ['Intervalo', 'Cuenta', 'Media', 'Desviación estándar', 'Mínimo', '25%', 'Mediana', '75%', 'Máximo']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_intervalos.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_intervalos[col] for col in tabla_intervalos.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas por Intervalos de presión arterial sistólica')
    fig.show()

    # Boxplot por intervalos categóricos
    intervalos_masculino = [0, 120, 130, 139, float('inf')]
    labels = ['[0, 120)', '[120, 130)', '[130, 139)', '[139, inf)']
    datos.datos['presion_arterial_sistolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_sistolica'],
        bins=intervalos_masculino,
        labels=labels,
        right=False,
        include_lowest=True
    )
    datos_filtrados = datos.datos.dropna(subset=['presion_arterial_sistolica_intervalo'])
    datos_filtrados['presion_arterial_sistolica_intervalo'] = pd.Categorical(
        datos_filtrados['presion_arterial_sistolica_intervalo'],
        categories=labels,
        ordered=True
    )
    fig = px.box(
        datos_filtrados, 
        x='presion_arterial_sistolica_intervalo', 
        y='presion_arterial_sistolica', 
        title='Boxplot de Presión Arterial Sistólica por Intervalos', 
        color_discrete_sequence=["blue"]
    )
    fig.update_layout(
        xaxis_title='Intervalo', 
        yaxis_title='Presión Arterial Sistólica',
        xaxis={'categoryorder': 'array', 'categoryarray': labels}
    )
    fig.show()

plot_all()



C:\Users\loren\AppData\Local\Temp\ipykernel_15056\2123956951.py:106: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



#### Presion arterial diastolica

In [13]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

max_diastolica = max(datos.datos['presion_arterial_diastolica'])
intervalos = {'masculino': [0, 80, 90, 120, max_diastolica], 'femenino': [0, 80, 90, 120, max_diastolica]}

def plot_all():
    # Boxplot crudo
    fig = px.box(datos.datos, y='presion_arterial_diastolica', title='Boxplot crudo de presión arterial diastólica')
    fig.update_layout(yaxis_title='Presión Arterial Diastólica')
    fig.show()
    
    # Histograma crudo
    fig = px.histogram(datos.datos, x='presion_arterial_diastolica', nbins=10, 
                       title='Histograma crudo de presión arterial diastólica', marginal='rug')
    fig.update_layout(xaxis_title='Presión Arterial Diastólica')
    fig.show()
    
    # Tabla descriptiva general
    tabla = datos.datos['presion_arterial_diastolica'].describe().reset_index()
    tabla.columns = ['Estadística', 'Valor']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla[col] for col in tabla.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas de presión arterial diastólica')
    fig.show()
    
    # Tablas descriptivas por agrupaciones independientes
    agrupaciones = ['sexo', 'tipo_de_afiliacion', 
                    'escolaridad', 'diabetes mellitus', 'hipercolesterolemia', 'dislipidemia',
                    'no_diabeticos_POC_hba1c(%)_controlado',
    'diabeticos_POC_hba1c(%)_controlado', 
    'bajo_peso',
    'normo_peso',
    'sobrepeso',
    'obesidad',
    'perimetro_abdominal_normal',
    'perimetro_abdominal_aumentado',
    'perimetro_abdominal_incremento_sustancial',
    'tension_normal',
    'tension_elevada',
    'tension_alta_nivel_1',
    'tension_alta_nivel_2']
    for agrupacion in agrupaciones:
        tabla_agrupacion = datos.datos.groupby(agrupacion)['presion_arterial_diastolica'].describe().reset_index()
        tabla_agrupacion = tabla_agrupacion.rename(columns={
            'mean': 'Media', 'std': 'Desviación estándar', '50%': 'Mediana', 
            'min': 'Mínimo', 'max': 'Máximo'
        })
        fig = go.Figure(data=[go.Table(
            header=dict(values=list(tabla_agrupacion.columns), fill_color='paleturquoise', align='left'),
            cells=dict(values=[tabla_agrupacion[col] for col in tabla_agrupacion.columns], fill_color='lavender', align='left')
        )])
        fig.update_layout(title_text=f'Estadísticas Descriptivas por {agrupacion}')
        fig.show()
    
    # Tabla descriptiva por intervalos
    datos.datos['presion_arterial_diastolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_diastolica'], intervalos['masculino'], right=False
    ).astype(str)
    tabla_intervalos = datos.datos.groupby('presion_arterial_diastolica_intervalo')['presion_arterial_diastolica'].describe().reset_index()
    tabla_intervalos.columns = ['Intervalo', 'Cuenta', 'Media', 'Desviación estándar', 'Mínimo', '25%', 'Mediana', '75%', 'Máximo']
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla_intervalos.columns), fill_color='paleturquoise', align='left'),
        cells=dict(values=[tabla_intervalos[col] for col in tabla_intervalos.columns], fill_color='lavender', align='left')
    )])
    fig.update_layout(title_text='Estadísticas Descriptivas por Intervalos de presión arterial diastólica')
    fig.show()
    
    # Boxplot por intervalos categóricos
    intervalos_diastolica = [0, 80, 90, 100, float('inf')]
    labels_diastolica = ['[0, 80)', '[80, 90)', '[90, 100)', '[100, inf)']
    datos.datos['presion_arterial_diastolica_intervalo'] = pd.cut(
        datos.datos['presion_arterial_diastolica'], bins=intervalos_diastolica, 
        labels=labels_diastolica, right=False, include_lowest=True
    )
    datos_filtrados_diastolica = datos.datos.dropna(subset=['presion_arterial_diastolica_intervalo'])
    datos_filtrados_diastolica['presion_arterial_diastolica_intervalo'] = pd.Categorical(
        datos_filtrados_diastolica['presion_arterial_diastolica_intervalo'], 
        categories=labels_diastolica, ordered=True
    )
    fig = px.box(datos_filtrados_diastolica, x='presion_arterial_diastolica_intervalo', 
                 y='presion_arterial_diastolica', title='Boxplot de Presión Arterial Diastólica por Intervalos')
    fig.update_layout(
        xaxis_title='Intervalo', yaxis_title='Presión Arterial Diastólica', 
        xaxis={'categoryorder': 'array', 'categoryarray': labels_diastolica}
    )
    fig.show()

plot_all()



C:\Users\loren\AppData\Local\Temp\ipykernel_15056\606570040.py:80: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



## Conteos

In [14]:
for agrupacion in ['lugar_de_procedencia', 'lugar_de_nacimiento','tipo_de_afiliacion','escolaridad']:
            conteo_ciudades = datos.datos[agrupacion].value_counts().head(10)  
            df_ciudades = conteo_ciudades.reset_index()
            df_ciudades.columns = [agrupacion, 'Conteo']  
            fig = px.bar(df_ciudades, x=agrupacion, y='Conteo', title=f'10 Conteos con mas Más Apariciones por {agrupacion}', 
                        color='Conteo', text='Conteo', color_continuous_scale='blues')
            fig.update_layout(xaxis_title=agrupacion, yaxis_title='Número de Apariciones', xaxis_tickangle=-45)
            fig.show()